# CIFAR-10 teacher ResNet for knowledge distillation
Train this larger teacher from scratch on Kaggle with a GPU. The teacher is an intermediate training tool; the final submitted student must satisfy the 5M parameter limit. No pretrained weights or external datasets are used.

The teacher uses stages with 128, 256, and 512 channels, three blocks per stage, and approximately 17.3M trainable parameters. Keep the same fixed stratified 80/20 split and class mapping for student experiments. The teacher must not train on validation images while that split is used to evaluate the student.

Training saves `best_teacher.pt`, `teacher_history.csv`, and `teacher_split.csv`. Preserve these Kaggle outputs for the student notebook. If GPU memory is insufficient, reduce `BATCH_SIZE` to 64; do not change the validation split.

Optimized execution settings for Kaggle T4 ×2 are described in the training section.


In [ ]:
from pathlib import Path
import csv
import os
import time
import random
import numpy as np
from PIL import Image
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

SEED = 42
BATCH_SIZE = 128
EPOCHS = 160
LEARNING_RATE = 0.1
NUM_WORKERS = min(4, os.cpu_count() or 1)
USE_TWO_GPUS = True
USE_AMP = True
FAST_CUDNN = True  # Faster convolution selection; disables strict determinism.
TEACHER_CONFIG = dict(channels=(128, 256, 512), blocks_per_stage=3, num_classes=10)
INPUT_ROOT = Path('/kaggle/input/competitions/cifar-10-res-net-architecture-challenge-dl')
WORK_DIR = Path('/kaggle/working')
WORK_DIR.mkdir(parents=True, exist_ok=True)
LABELS_CSV = INPUT_ROOT / 'train_labels.csv'
TRAIN_IMAGE_DIR = INPUT_ROOT / 'train_images'

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = FAST_CUDNN
    torch.backends.cudnn.deterministic = not FAST_CUDNN

seed_everything(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
AMP_ENABLED = USE_AMP and device.type == 'cuda'
GPU_IDS = list(range(min(2 if USE_TWO_GPUS else 1, torch.cuda.device_count())))
print('GPU count used:', len(GPU_IDS), '| FP16 AMP:', AMP_ENABLED)
for gpu_id in GPU_IDS:
    print(gpu_id, torch.cuda.get_device_name(gpu_id))
print('Global batch size:', BATCH_SIZE)


## Load labels and freeze the split
Expected data: a CSV with `id,label` columns and one image per ID (for example `train_images/train_00001.png`). Class names are mapped in sorted order and saved. The split uses a dedicated NumPy generator, independent of training randomness. Each class must have a sample count divisible by five so the requested 80/20 ratio is exact within every class.


In [ ]:
assert LABELS_CSV.is_file(), f'Missing labels CSV: {LABELS_CSV}'
assert TRAIN_IMAGE_DIR.is_dir(), f'Missing training image directory: {TRAIN_IMAGE_DIR}'
LABELS_CSV = Path(LABELS_CSV)
with LABELS_CSV.open(newline='') as f:
    reader = csv.DictReader(f)
    assert {'id', 'label'} <= set(reader.fieldnames or []), 'Expected id,label columns'
    rows = sorted(list(reader), key=lambda row: row['id'])
assert rows and len({r['id'] for r in rows}) == len(rows), 'IDs must be unique'
classes = sorted({r['label'] for r in rows})
assert len(classes) == 10, classes
class_to_idx = {name: i for i, name in enumerate(classes)}
labels = np.array([class_to_idx[r['label']] for r in rows], dtype=np.int64)
TRAIN_IMAGE_DIR = Path(TRAIN_IMAGE_DIR)
def training_image_path(image_id):
    # Accept numeric IDs, train_00001 IDs, or train_00001.png filenames.
    stem = Path(str(image_id).strip()).stem
    if stem.startswith('train_'):
        stem = stem[len('train_'):]
    if not stem.isdigit():
        raise ValueError(f'Unexpected training image ID: {image_id!r}')
    return TRAIN_IMAGE_DIR / f'train_{int(stem):05d}.png'

image_paths = [training_image_path(r['id']) for r in rows]
missing = [str(p) for p in image_paths if not p.is_file()]
assert not missing, f'Missing images, first examples: {missing[:5]}'

rng = np.random.default_rng(SEED)
train_indices, val_indices = [], []
for cls in range(10):
    indices = rng.permutation(np.flatnonzero(labels == cls))
    assert len(indices) % 5 == 0, 'Exact stratified 80/20 requires class sizes divisible by 5'
    n_val = len(indices) // 5
    val_indices.extend(indices[:n_val].tolist())
    train_indices.extend(indices[n_val:].tolist())
train_indices = np.array(sorted(train_indices))
val_indices = np.array(sorted(val_indices))
assert len(train_indices) * 5 == len(rows) * 4
assert len(val_indices) * 5 == len(rows)
assert not set(train_indices) & set(val_indices)
assert set(train_indices) | set(val_indices) == set(range(len(rows)))
with (WORK_DIR / 'teacher_split.csv').open('w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['id', 'label', 'split'])
    val_set = set(val_indices)
    for i, row in enumerate(rows):
        writer.writerow([row['id'], row['label'], 'validation' if i in val_set else 'training'])
print('Class mapping:', class_to_idx)
print('Training / validation:', len(train_indices), len(val_indices))
for cls, name in enumerate(classes):
    print(name, int((labels[train_indices] == cls).sum()), int((labels[val_indices] == cls).sum()))

In [ ]:
# Fixed CIFAR normalization constants; no statistics are fitted on validation data.
MEAN = (0.4914, 0.4822, 0.4465)
STD = (0.2470, 0.2435, 0.2616)
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class LabeledImages(Dataset):
    def __init__(self, indices, transform):
        self.indices = np.asarray(indices)
        self.transform = transform
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, index):
        i = int(self.indices[index])
        with Image.open(image_paths[i]) as image:
            image = image.convert('RGB')
            assert image.size == (32, 32), f'Unexpected image size: {image.size}'
            x = self.transform(image)
        return x, int(labels[i])

train_dataset = LabeledImages(train_indices, train_transform)
val_dataset = LabeledImages(val_indices, val_transform)
assert torch.equal(val_dataset[0][0], val_dataset[0][0]), 'Validation must be deterministic'

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

# Persistent workers avoid restarting worker processes every epoch.
loader_options = dict(num_workers=NUM_WORKERS, pin_memory=device.type == 'cuda')
if NUM_WORKERS > 0:
    loader_options.update(persistent_workers=True, prefetch_factor=2)
# A dedicated generator fixes shuffle order; fast cuDNN is not bitwise deterministic.
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    worker_init_fn=seed_worker, **loader_options,
    generator=torch.Generator().manual_seed(SEED),
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    **loader_options,
)


## Larger teacher architecture
A CIFAR-style 3×3 stem feeds three residual stages with 128, 256, and 512 channels and three blocks per stage. Global average pooling feeds the classifier. The 5M limit is enforced in the final student notebook, not in this intermediate teacher notebook.


In [ ]:
class BasicBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.residual = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
        )
        self.shortcut = nn.Identity() if stride == 1 and in_channels == out_channels else nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
            nn.BatchNorm2d(out_channels),
        )
        self.relu = nn.ReLU(inplace=True)
    def forward(self, x):
        return self.relu(self.residual(x) + self.shortcut(x))

class CIFARResNet(nn.Module):
    def __init__(self, channels=(64, 128, 256), blocks_per_stage=3, num_classes=10):
        super().__init__()
        assert len(channels) == 3 and blocks_per_stage >= 1
        c1, c2, c3 = channels
        self.stem = nn.Sequential(
            nn.Conv2d(3, c1, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(c1),
            nn.ReLU(inplace=True),
        )
        def stage(in_channels, out_channels, stride):
            return nn.Sequential(
                BasicBlock(in_channels, out_channels, stride),
                *[BasicBlock(out_channels, out_channels) for _ in range(blocks_per_stage - 1)],
            )
        self.stage1 = stage(c1, c1, 1)
        self.stage2 = stage(c1, c2, 2)
        self.stage3 = stage(c2, c3, 2)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(c3, num_classes)
        for module in self.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.kaiming_normal_(module.weight, mode='fan_out', nonlinearity='relu')
    def forward(self, x):
        x = self.stage3(self.stage2(self.stage1(self.stem(x))))
        return self.classifier(self.pool(x).flatten(1))

model = CIFARResNet(**TEACHER_CONFIG)
parameter_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
for name, module in model.named_children():
    print(f'{name}: {sum(p.numel() for p in module.parameters() if p.requires_grad):,}')
print(f'Trainable parameters: {parameter_count:,}')
print('Teacher only: final student must have at most 5,000,000 parameters.')
with torch.no_grad():
    model.eval()
    assert model(torch.zeros(2, 3, 32, 32)).shape == (2, 10)

## Train for 160 epochs
Cross entropy with SGD (learning rate 0.1, momentum 0.9, weight decay 5e-4), batch size 128, and cosine decay stepped once after each epoch. Training uses padded random crops and horizontal flips. Validation uses evaluation mode and deterministic preprocessing. Save the best validation checkpoint and per-epoch metrics in `/kaggle/working`.

### Kaggle T4 ×2 execution
Select GPU T4 ×2 in Kaggle and restart the session before running all cells. This notebook uses up to two GPUs via DataParallel and FP16 autocast with gradient scaling, falling back to one GPU or CPU when necessary. The global batch stays 128 (64 per GPU on two GPUs), preserving the original learning-rate recipe. Try global batch 256 only as a measured experiment; larger batches can change convergence.

Set `USE_TWO_GPUS=False` to benchmark one GPU. Compare epoch times after the first warm-up epoch: DataParallel replication and communication can limit scaling for small images. DistributedDataParallel is the next option for maximum throughput, but requires a separate multi-process training script. Set `FAST_CUDNN=False` for deterministic cuDNN settings; worker seeds and a fixed split alone do not guarantee bitwise-identical GPU runs. Checkpoints save the unwrapped model, so the distillation notebook needs no parallel wrapper.


In [ ]:
# Fresh weights for the full run; the diagnostic model is not reused.
seed_everything(SEED)
train_loader.generator.manual_seed(SEED)
# Autocast lives inside forward because DataParallel executes GPU replicas in threads.
class AMPModel(nn.Module):
    def __init__(self, network):
        super().__init__()
        self.network = network
    def forward(self, images):
        with torch.autocast(device_type=device.type, dtype=torch.float16,
                            enabled=AMP_ENABLED):
            return self.network(images)

base_model = CIFARResNet(**TEACHER_CONFIG).to(device)
model = AMPModel(base_model)
if len(GPU_IDS) > 1:
    model = nn.DataParallel(model, device_ids=GPU_IDS)
scaler = torch.amp.GradScaler('cuda', enabled=AMP_ENABLED)
parameter_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
# The intermediate teacher is not subject to the student's parameter budget.
print(f'Trainable parameters: {parameter_count:,}')
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(
    model.parameters(), lr=LEARNING_RATE, momentum=0.9, weight_decay=5e-4,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)


def train_epoch():
    model.train()
    loss_sum = torch.zeros((), device=device)
    correct = torch.zeros((), dtype=torch.long, device=device)
    count = 0
    for images, targets in train_loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits.float(), targets)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        loss_sum += loss.detach() * targets.size(0)
        correct += (logits.detach().argmax(1) == targets).sum()
        count += targets.size(0)
    mean_loss = (loss_sum / count).item()
    assert np.isfinite(mean_loss), 'Non-finite epoch loss'
    return mean_loss, correct.item() / count


@torch.inference_mode()
def evaluate():
    model.eval()
    loss_sum = torch.zeros((), device=device)
    correct = torch.zeros((), dtype=torch.long, device=device)
    count = 0
    for images, targets in val_loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        logits = model(images)
        loss_sum += criterion(logits.float(), targets) * targets.size(0)
        correct += (logits.argmax(1) == targets).sum()
        count += targets.size(0)
    mean_loss = (loss_sum / count).item()
    assert np.isfinite(mean_loss), 'Non-finite epoch loss'
    return mean_loss, correct.item() / count


best_accuracy = -1.0
best_epoch = 0
history = []
for epoch in range(1, EPOCHS + 1):
    epoch_start = time.perf_counter()
    learning_rate = optimizer.param_groups[0]['lr']
    train_loss, train_accuracy = train_epoch()
    val_loss, val_accuracy = evaluate()
    scheduler.step()
    epoch_seconds = time.perf_counter() - epoch_start
    metrics = dict(epoch=epoch, learning_rate=learning_rate, epoch_seconds=epoch_seconds,
                   train_loss=train_loss, train_accuracy=train_accuracy,
                   val_loss=val_loss, val_accuracy=val_accuracy)
    history.append(metrics)
    if val_accuracy > best_accuracy:
        best_accuracy, best_epoch = val_accuracy, epoch
        torch.save(dict(
            model_state=base_model.state_dict(), epoch=epoch,
            model_config=TEACHER_CONFIG, role='teacher',
            execution_config=dict(amp=AMP_ENABLED, gpu_count=len(GPU_IDS),
                                  batch_size=BATCH_SIZE, fast_cudnn=FAST_CUDNN),
            class_to_idx=class_to_idx,
            val_accuracy=val_accuracy, parameter_count=parameter_count,
            classes=classes, seed=SEED,
            train_indices=train_indices.tolist(), val_indices=val_indices.tolist(),
            normalization=dict(mean=MEAN, std=STD),
        ), WORK_DIR / 'best_teacher.pt')
    # Flush history every epoch so metrics survive an interrupted Kaggle session.
    with (WORK_DIR / 'teacher_history.csv').open('w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=metrics.keys())
        writer.writeheader()
        writer.writerows(history)
    print(f'Epoch {epoch:03d}/{EPOCHS}: lr={learning_rate:.6f}, '
          f'train loss={train_loss:.4f}, train acc={train_accuracy:.2%}, '
          f'val loss={val_loss:.4f}, val acc={val_accuracy:.2%}, '
          f'best={best_accuracy:.2%}, seconds={epoch_seconds:.1f}', flush=True)

# Leave the best model loaded for subsequent evaluation or test prediction.
checkpoint = torch.load(WORK_DIR / 'best_teacher.pt', map_location=device, weights_only=True)
base_model.load_state_dict(checkpoint['model_state'])
model.eval()
print(f'Best validation accuracy: {best_accuracy:.2%} (epoch {best_epoch})')
print(f'Trainable parameters: {parameter_count:,}')
print('Saved:', WORK_DIR / 'best_teacher.pt', WORK_DIR / 'teacher_history.csv')


## Use this checkpoint for student distillation
Copy the `CIFARResNet` and `BasicBlock` definitions into the student notebook, reconstruct the teacher with `CIFARResNet(**checkpoint['model_config'])`, and load `checkpoint['model_state']`. Set `teacher.eval()` and `teacher.requires_grad_(False)`.

Verify that class order, normalization, and split IDs match the checkpoint before training. Feed the same augmented image tensor to both teacher and student. Run the frozen teacher under `torch.no_grad()` and train the student using a weighted combination of cross entropy on labels and temperature-scaled KL divergence on teacher predictions. Start with temperature 4 and soft-target weight 0.5, then compare against the supervised student baseline on the same validation split. These settings are starting points, not guaranteed improvements.

Only the student is needed for final inference. Enforce its 5M parameter limit and select its best checkpoint by validation accuracy. Do not use teacher predictions on held-out validation images as student training targets.
